## Importación de librerías

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

ne2_imp = pd.read_csv('../data_estaciones/BD_NE2_imputado_completo.csv', parse_dates=['time'])
ne3_imp = pd.read_csv('../data_estaciones/BD_NE3_imputado_completo.csv', parse_dates=['time'])
ne2_raw = pd.read_csv('../data_estaciones/BD_NE2_limpia.csv', parse_dates=['date']).rename(columns={'date':'time'})
ne3_raw = pd.read_csv('../data_estaciones/BD_NE3_limpia.csv', parse_dates=['date']).rename(columns={'date':'time'})
ndvi = pd.read_csv('../data_estaciones/ndvi_mensual_estaciones_2020_2025.csv', parse_dates=['mes'])

## Análisis exploratorio

In [ ]:
import pandas as pd

DATA = '../data_estaciones'
ndvi = pd.read_csv(f'{DATA}/ndvi_mensual_estaciones_2020_2025.csv', parse_dates=['mes'])

# NDVI mediano por estación
print("=== NDVI mediano por estación ===")
for est in ['NE2', 'NE3']:
    mediana = ndvi[ndvi.estacion == est]['ndvi_mediana'].median()
    print(f'{est}: NDVI mediano = {mediana:.3f}')

# Correlación NDVI-PM10: imputado vs. sin imputar
def monthly_raw(estacion, inicio='2021-01-01', fin='2025-06-01'):
    df = pd.read_csv(f'{DATA}/BD_{estacion}_limpia.csv', parse_dates=['date'])
    g = df.set_index('date')['PM10'].resample('MS').median()
    out = pd.DataFrame({'mes': g.index, 'PM10': g.values})
    return out[(out.mes >= inicio) & (out.mes <= fin)]

def monthly_imp(estacion, inicio='2021-01-01', fin='2025-06-01'):
    df = pd.read_csv(f'{DATA}/BD_{estacion}_imputado_completo.csv', parse_dates=['time'])
    g = df.set_index('time')['PM10'].resample('MS').median()
    out = pd.DataFrame({'mes': g.index, 'PM10': g.values})
    return out[(out.mes >= inicio) & (out.mes <= fin)]

print("\n=== Correlación (Spearman) NDVI-PM10: imputado vs. sin imputar ===")
for est in ['NE2', 'NE3']:
    nd_est = ndvi[ndvi.estacion == est][['mes', 'ndvi_mediana']]
    raw = monthly_raw(est).merge(nd_est, on='mes')
    imp = monthly_imp(est).merge(nd_est, on='mes')
    r_raw = raw['PM10'].corr(raw['ndvi_mediana'], method='spearman')
    r_imp = imp['PM10'].corr(imp['ndvi_mediana'], method='spearman')
    print(f'{est}: sin imputar (n={len(raw)}) r={r_raw:.3f} | imputado (n={len(imp)}) r={r_imp:.3f}')

=== NDVI mediano por estación ===
NE2: NDVI mediano = 0.051
NE3: NDVI mediano = 0.198

=== Correlación (Spearman) NDVI-PM10: imputado vs. sin imputar ===
NE2: sin imputar (n=54) r=0.101 | imputado (n=54) r=0.113
NE3: sin imputar (n=54) r=-0.210 | imputado (n=54) r=-0.192


## Regresión con efectos fijos de mes para usar el año completo

  Se le agregan al modelo variables "dummy" (una por cada mes calendario) que absorben el promedio esperado de cada mes. Esto dejar solo la variación real (la desviación de cada mes/semana respecto a lo típico de esa época), sin necesitar restringir la muestra.

In [4]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

DATA = '../data_estaciones'
ndvi_mensual = pd.read_csv(f'{DATA}/ndvi_mensual_estaciones_2020_2025.csv', parse_dates=['mes'])
evi = pd.read_csv(f'{DATA}/EVI_estaciones.csv', parse_dates=['fecha'])

# series mensuales (PM10, RH, índice de vegetación)
def construir_mensual(estacion, indice='ndvi'):
    """Agrega PM10 (mediana) y RH (promedio) a nivel mensual, y las une con
    el índice de vegetación elegido (NDVI o EVI)."""
    df = pd.read_csv(f'{DATA}/BD_{estacion}_limpia.csv', parse_dates=['date'])
    df = df[(df.date >= '2021-01-01') & (df.date <= '2025-12-31')]
    g = df.set_index('date')
    pm10 = g['PM10'].resample('MS').median()
    rh = g['RH'].resample('MS').mean()
    mensual = pd.DataFrame({'mes': pm10.index, 'PM10': pm10.values, 'RH': rh.values})

    if indice == 'ndvi':
        ind = ndvi_mensual[ndvi_mensual.estacion == estacion][['mes', 'ndvi_mediana']]
        ind = ind.rename(columns={'ndvi_mediana': 'indice_veg'})
    else:  # evi, compuesto de 16 dias -> se promedia a nivel mensual
        e = evi[evi.estacion == estacion].copy()
        e['mes'] = e.fecha.dt.to_period('M').dt.to_timestamp()
        ind = e.groupby('mes')['EVI'].mean().reset_index().rename(columns={'EVI': 'indice_veg'})

    d = mensual.merge(ind, on='mes').dropna().sort_values('mes').reset_index(drop=True)
    d['mesnum'] = d.mes.dt.month  # 1=enero, ..., 12=diciembre
    return d

# modelo con efectos fijos de mes + errores robustos
def modelo_efectos_fijos(d, y_col, x_cols, maxlags=4):
    """
    Regresion OLS de y_col sobre x_cols, controlando el mes calendario con
    variables dummy (drop_first=True evita colinealidad perfecta: un mes
    queda como referencia implícita).

    Se usan errores estándar HAC (Newey-West) porque observaciones
    consecutivas en el tiempo suelen estar correlacionadas entre sí
    (autocorrelación); ignorarlo produce p-valores artificialmente
    pequeños. maxlags=4 para datos mensuales, se sube a 8 para semanales.
    """
    dummies = pd.get_dummies(d['mesnum'], prefix='m', drop_first=True).astype(float)
    X = pd.concat([d[x_cols].reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)
    X = sm.add_constant(X)
    modelo = sm.OLS(d[y_col].reset_index(drop=True), X).fit(cov_type='HAC', cov_kwds={'maxlags': maxlags})
    return modelo

# Correr los tres modelos de la cadena propuesta:
# (a) índice de vegetación -> humedad relativa (RH)
# (b) RH -> PM10, controlando el índice de vegetación
# (c) índice de vegetación -> PM10, controlando RH (efecto directo residual)
resultados = []
for indice in ['ndvi', 'evi']:
    for est in ['NE2', 'NE3']:
        d = construir_mensual(est, indice)

        m_rh = modelo_efectos_fijos(d, 'RH', ['indice_veg'], maxlags=4)
        m_pm10 = modelo_efectos_fijos(d, 'PM10', ['indice_veg', 'RH'], maxlags=4)

        resultados.append({
            'indice': indice.upper(), 'estacion': est, 'n': len(d),
            'coef_indice_RH': round(m_rh.params['indice_veg'], 3),
            'p_indice_RH': round(m_rh.pvalues['indice_veg'], 4),
            'coef_RH_PM10': round(m_pm10.params['RH'], 3),
            'p_RH_PM10': round(m_pm10.pvalues['RH'], 4),
            'coef_indice_PM10': round(m_pm10.params['indice_veg'], 2),
            'p_indice_PM10': round(m_pm10.pvalues['indice_veg'], 4),
            'R2_modelo_PM10': round(m_pm10.rsquared, 3),
        })

tabla = pd.DataFrame(resultados)
print(tabla.to_string(index=False))

# Repetir a nivel semanal
def construir_semanal(estacion):
    """Usa el archivo semanal ya construido (PM10, NDVI, RH) sin valores nulos."""
    df = pd.read_csv(f'{DATA}/{estacion}_semanal_mediana_con_ndvi_sin_nulos.csv', parse_dates=['semana'])
    df = df[(df.semana >= '2021-01-01') & (df.semana <= '2025-12-31')]
    d = df[['semana', 'PM10', 'RH', 'ndvi_mediana']].dropna().reset_index(drop=True)
    d = d.rename(columns={'ndvi_mediana': 'indice_veg', 'semana': 'mes'})  # renombrar para reusar la funcion
    d['mesnum'] = d['mes'].dt.month
    return d

print("\n=== SEMANAL (n≈230, maxlags=8 por mayor autocorrelación semana a semana) ===")
for est in ['NE2', 'NE3']:
    d = construir_semanal(est)
    m_rh = modelo_efectos_fijos(d, 'RH', ['indice_veg'], maxlags=8)
    m_pm10 = modelo_efectos_fijos(d, 'PM10', ['indice_veg', 'RH'], maxlags=8)
    print(f'\n--- NDVI {est} semanal (n={len(d)}) ---')
    print(f'  NDVI -> RH:  coef={m_rh.params["indice_veg"]:.2f}, p={m_rh.pvalues["indice_veg"]:.4f}')
    print(f'  RH -> PM10:  coef={m_pm10.params["RH"]:.3f}, p={m_pm10.pvalues["RH"]:.4f}')
    print(f'  NDVI -> PM10: coef={m_pm10.params["indice_veg"]:.2f}, p={m_pm10.pvalues["indice_veg"]:.4f}')

indice estacion  n  coef_indice_RH  p_indice_RH  coef_RH_PM10  p_RH_PM10  coef_indice_PM10  p_indice_PM10  R2_modelo_PM10
  NDVI      NE2 60          -3.276       0.9554        -0.567     0.0354            197.05         0.0792           0.458
  NDVI      NE3 60           5.083       0.4156        -0.471     0.0000              6.61         0.4760           0.531
   EVI      NE2 60         108.227       0.0016        -0.805     0.0002            182.58         0.0163           0.458
   EVI      NE3 60          63.046       0.0001        -0.609     0.0006             33.99         0.1414           0.549

=== SEMANAL (n≈230, maxlags=8 por mayor autocorrelación semana a semana) ===

--- NDVI NE2 semanal (n=231) ---
  NDVI -> RH:  coef=-136.76, p=0.0000
  RH -> PM10:  coef=-0.594, p=0.0000
  NDVI -> PM10: coef=57.14, p=0.1063

--- NDVI NE3 semanal (n=233) ---
  NDVI -> RH:  coef=-23.62, p=0.0004
  RH -> PM10:  coef=-0.575, p=0.0000
  NDVI -> PM10: coef=8.81, p=0.2158


RH->PM10 sigue siendo muy significativo (p<0.001) en ambas estaciones, confirma que es el hallazgo más robusto de todo el análisis, sin importar la resolución temporal.

RH -> PM10 (menor humedad, mayor PM10): 6/6 significativas, mismo signo, robusto a HAC

EVI -> RH (más vegetación, más humedad) mensual: significativo en ambas estaciones

NDVI -> RH -- mensual: no significativo en ninguna estación

NDVI -> RH -- semanal: significativo pero con signo contradictorio (ruido de medición)

Índice vegetación -> PM10 directo (controlando RH): no hay evidencia consistente de efecto directo
